### Historical "life" tracker

In [1]:
%matplotlib qt

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import livef1

# Get session
session = livef1.get_session(
    season=2026,
    meeting_identifier="Austria",
    session_identifier="Race"
)

session.generate()

telemetry = session.carTelemetry


# Drivers you want to display
driver_numbers = ["81", "1", "44", "16"]
driver_numbers = telemetry["DriverNo"].unique().tolist()


# Prepare data
drivers = {}

for driver in driver_numbers:
    df = telemetry[telemetry["DriverNo"] == driver].copy()
    df = df.sort_values("timestamp").reset_index(drop=True)
    drivers[driver] = df


# -------------------------
# Plot setup
# -------------------------

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect("equal")


# Draw track using all cars
track = telemetry[
    (telemetry["X"] != 0) &
    (telemetry["Y"] != 0)
]

ax.plot(
    track["X"],
    track["Y"],
    color="gray",
    alpha=0.3
)


# Find limits
ax.set_xlim(track["X"].min()-100, track["X"].max()+100)
ax.set_ylim(track["Y"].min()-100, track["Y"].max()+100)


# Create points for every driver
points = {}
labels = {}

for driver in driver_numbers:

    points[driver], = ax.plot(
        [],
        [],
        "o",
        markersize=8,
        label=driver
    )

    labels[driver] = ax.text(
        0,
        0,
        driver,
        fontsize=10
    )


ax.legend()


# -------------------------
# Animation
# -------------------------

def update(frame):

    for driver in driver_numbers:

        df = drivers[driver]

        # protect if driver has fewer telemetry points
        if frame >= len(df):
            continue

        row = df.iloc[frame]

        x = row["X"]
        y = row["Y"]

        points[driver].set_data(
            [x],
            [y]
        )

        labels[driver].set_position(
            (x + 20, y + 20)
        )


    return list(points.values()) + list(labels.values())


ani = FuncAnimation(
    fig,
    update,
    frames=max(len(df) for df in drivers.values()),
    interval=20,
    blit=True
)


plt.show()

22:39:18 - Driver standings have been loaded and saved to 'season.driverStandings'.
22:39:18 - Constructor standings have been loaded and saved to 'season.constructorStandings'.
22:39:18 - The identifier couldn't be found.
22:39:18 - The identifier is very close to 'Austrian Grand Prix' at column 'MEETING NAME'
22:39:18 - Selected meeting/session is:
	Meeting Offname : FORMULA 1 LENOVO AUSTRIAN GRAND PRIX 2026
	Meeting Name : Austrian Grand Prix
	Meeting Circuit Shortname : Spielberg
22:39:18 - Got the meeting.
22:39:18 - Selected meeting/session is:
	session_name : Race
22:39:18 - The session was received successfully.
22:39:19 - Fetching drivers.
22:39:19 - Driver standings have been loaded and saved to 'session.driverStandings'.
22:39:19 - Constructor standings have been loaded and saved to 'session.constructorStandings'.
22:39:20 - Session results have been loaded and saved to 'session.sessionResults'.
22:39:20 - Starting grid have been loaded and saved to 'session.startingGrid'.
2

### Historical "life" tracker /w extra information

In [ ]:
%matplotlib qt

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import livef1


# --- load data ---

session = livef1.get_session(
    season=2026,
    meeting_identifier="Spa",
    session_identifier="Race"
)
session.generate()

telemetry = session.carTelemetry

driver_info = session.get_data(dataNames="DriverList")

# One dataframe per driver, sorted by time
drivers = {
    driver: df.sort_values("timestamp").reset_index(drop=True)
    for driver, df in telemetry.groupby("DriverNo")
}
driver_numbers = list(drivers.keys())


# --- Figure layout (GridSpec via subplots) ---


fig, (left_ax, track_ax, info_ax) = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(16, 8),
    gridspec_kw={"width_ratios": [1, 3, 1]}
)

fig.subplots_adjust(left=0.04, right=0.98, top=0.95, bottom=0.05, wspace=0.15)

track_ax.set_aspect("equal", adjustable="datalim")
left_ax.axis("off")
info_ax.axis("off")

# TODO: Add new info here
left_ax.set_title("Standings / extra info here")

# TODO: find real track data
# --- Draw track ---

track = telemetry[(telemetry["X"] != 0) & (telemetry["Y"] != 0)]

# Scatter, not plot: `track` mixes every driver's every lap, so connecting
# consecutive rows with lines jumps between unrelated cars/laps and draws
# stray diagonal lines across the circuit. Individual points still trace
# a clean track outline given how dense the telemetry samples are.
track_ax.scatter(
    track["X"], track["Y"],
    s=1, color="gray", alpha=0.2, linewidths=0
)
track_ax.set_xlim(track["X"].min() - 200, track["X"].max() + 200)
track_ax.set_ylim(track["Y"].min() - 200, track["Y"].max() + 200)



# --- Create cars + labels ---

points = {
    driver: track_ax.plot([], [], "o", markersize=8, label=driver)[0]
    for driver in driver_numbers
}

labels = {
    driver: track_ax.text(0, 0, driver, fontsize=8)
    for driver in driver_numbers
}

# --- Telemetry table ---

n_drivers = len(driver_numbers)

columns = [
    ("pos",    "Pos",    0.02, "left"),
    ("driver", "Driver", 0.16, "left"),
    ("speed",  "Speed",  0.60, "right"),
    ("tire",   "Tire",   0.85, "left"),
]

# Fit exactly (n_drivers + header) rows into info_ax's actual height.
info_ax_height_in = fig.get_size_inches()[1] * info_ax.get_position().height
row_height = 1 / (n_drivers + 1)  # in axes-fraction units, +1 for header
fontsize = max(6, min(11, (info_ax_height_in * 72 * row_height) * 0.55))

header_y = 1 - row_height * 0.5

header_cells = {
    key: info_ax.text(
        x, header_y, label,
        fontsize=fontsize, fontweight="bold", family="monospace",
        ha=align, va="center", transform=info_ax.transAxes,
        clip_on=False
    )
    for key, label, x, align in columns
}

info_ax.axhline(
    header_y - row_height * 0.5, color="white", linewidth=0.5, alpha=0.4
)

# One row of cells per rank (P1 at top, down to last position).
# Which driver occupies a row changes every frame based on current position.
row_cells = []
for r in range(n_drivers):
    y = header_y - row_height * (r + 1)
    row_cells.append({
        key: info_ax.text(
            x, y, "",
            fontsize=fontsize, family="monospace", ha=align, va="center",
            transform=info_ax.transAxes, clip_on=False
        )
        for key, label, x, align in columns
    })


# --- Tire colors ---


tire_colors = {
    "SOFT": "red",
    "MEDIUM": "orange",
    "HARD": "white",
    "INTERMEDIATE": "green",
    "WET": "blue",
}



# --- Animation update ----


def update(frame):
    # Grab this frame's row per driver first (some drivers run out of data early)
    current_rows = {
        driver: drivers[driver].iloc[frame]
        for driver in driver_numbers
        if frame < len(drivers[driver])
    }

    # Sort drivers by current race position for display order only.
    # Car locations on the track are unaffected - this just controls
    # which table row each driver's data lands in.
    display_order = sorted(
        current_rows.keys(),
        key=lambda d: int(current_rows[d].get("Position", 999))
    )

    for driver in display_order:
        row = current_rows[driver]
        x, y = row["X"], row["Y"]

        points[driver].set_data([x], [y])
        labels[driver].set_position((x + 20, y + 20))

        compound = row.get("Compound", "UNKNOWN")
        points[driver].set_color(f"#{driver_info[driver_info['RacingNumber'] == row.get("DriverNo")]['TeamColour'].iloc[0]}")
        

    # Fill in the standings table, one row per current rank (r == 0 is P1)
    for r, cells in enumerate(row_cells):
        if r >= len(display_order):
            # No driver left for this rank this frame - blank the row
            for cell in cells.values():
                cell.set_text("")
            continue

        driver = display_order[r]
        row = current_rows[driver]
        compound = row.get("Compound", "UNKNOWN")

        cells["pos"].set_text(str(row.get("Position", "?")))
        cells["driver"].set_text(str(driver))
        cells["speed"].set_text(
            f"{round(row['Speed'], 1)}" if "Speed" in row else ""
        )
        cells["tire"].set_text(compound[0])
        cells["tire"].set_color(tire_colors.get(compound, "white"))

    return (
        list(points.values())
        + list(labels.values())
        + list(header_cells.values())
        + [cell for cells in row_cells for cell in cells.values()]
    )



# --- Start animation ---

ani = FuncAnimation(
    fig,
    update,
    frames=max(len(df) for df in drivers.values()),
    interval=20,
    blit=True
)

plt.show()

00:39:47 - Driver standings have been loaded and saved to 'season.driverStandings'.
00:39:47 - Constructor standings have been loaded and saved to 'season.constructorStandings'.
00:39:47 - Selected meeting/session is:
	Meeting Offname : FORMULA 1 MOËT & CHANDON BELGIAN GRAND PRIX 2026
	Meeting Name : Belgian Grand Prix
	Meeting Circuit Shortname : Spa-Francorchamps
00:39:47 - Got the meeting.
00:39:47 - Selected meeting/session is:
	session_name : Race
00:39:47 - The session was received successfully.
00:39:47 - Fetching drivers.
00:39:48 - Driver standings have been loaded and saved to 'session.driverStandings'.
00:39:48 - Constructor standings have been loaded and saved to 'session.constructorStandings'.
00:39:49 - Session results have been loaded and saved to 'session.sessionResults'.
00:39:49 - Starting grid have been loaded and saved to 'session.startingGrid'.
00:39:49 - The callback function for the SILVER table 'laps' was set.
00:39:49 - The callback function for the SILVER tabl

#00D7B6
#4781D7
#ED1131
#ED1131
#F47600
#6C98FF
#F50537
#6C98FF
#00A1E8
#00A1E8
#F50537
#F47600
#9C9FA2
#1868DB
#9C9FA2
#909090
#909090
#1868DB
#229971
#4781D7
#229971
